# OpenRouter 笔记本转换器

**第 4 周练习** — Mugisha Caleb Didier

## 练习目标（理念）

课程笔记本常为每个提供商（OpenAI、Anthropic、Google、xAI）各写一套直接 API 客户端，需要多把密钥。本工具用 **LLM 当代码转换器**：把任意课程笔记本里的「直接提供商 API」写法，批量改成走 **OpenRouter**（一把密钥、统一 `OpenAI` 兼容客户端）。

这是第 4 周同一模式（LLM as code converter）的不同目标：Direct-API Python → OpenRouter Python。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| LLM 改代码 | 一批带 `# ===CELL N===` 标记的单元格一次转换 |
| OpenRouter | `base_url="https://openrouter.ai/api/v1"` |
| 正则扫描 | `CONVERSION_PATTERNS` / `SKIP_PATTERNS` 决定改哪些格 |
| Gradio 工作流 | Scan → Convert → Apply（`.bak` 备份） |

## 怎么跑

1. 同目录有 `model_map.py`（`MODEL_MAP`）；`.env` 配好 `OPENROUTER_API_KEY`
2. 从上到下运行；最后 `ui.launch()` 打开界面
3. 选课笔记本 → **Scan** 预览 → **Convert** 看差异 → **Apply** 写回（先备份）


In [ ]:
# ========== 导入：文件 / JSON / 正则 / OpenAI / Gradio / 模型映射表 ==========

# 标准库 os：读环境变量（Environment Variables）
import os
# 标准库 json：读写 .ipynb（本质是 JSON）
import json
# 标准库 re：用正则判断单元格是否需要转换、拆分 LLM 输出
import re
# 标准库 copy：deepcopy 笔记本，避免改原对象
import copy
# 标准库 shutil：Apply 时 copy2 做 .bak 备份
import shutil
# Path：用面向对象路径拼 PROJECT_ROOT、扫描 week* 目录
from pathlib import Path
# load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# OpenAI 客户端：经 base_url 指向 OpenRouter
from openai import OpenAI
# gradio：Scan / Convert / Apply 的交互 UI
import gradio as gr
# 本地 model_map：友好名/裸 id → OpenRouter 带前缀 model id
from model_map import MODEL_MAP


In [ ]:
# ========== 环境与客户端：OpenRouter + 用于转换的大模型 ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)

# 读取 OpenRouter 密钥（字符串名保持原样）
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
# 有密钥则打印前 8 位确认；否则提示去 .env 添加（提示文案保持原样）
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set -- add OPENROUTER_API_KEY to your .env")

# 创建指向 OpenRouter 的兼容客户端
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)
# 负责「改代码」的模型 id（保持原样）
MODEL = "anthropic/claude-opus-4-6"


## 核心逻辑

关键设计决策：单元格**互相依赖**（密钥加载、客户端创建、模型字典彼此引用），所以把所有待转换代码格打成一批，用明确的 `# ===CELL N===` 标记送给 LLM；模型返回整批转换结果后，再按标记拆回各个单元格。


In [ ]:
# ========== 笔记本 I/O：读 JSON、取/写 cell source、写回磁盘 ==========

def read_notebook(path):
    # 以 UTF-8 打开 .ipynb，解析成 dict
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def get_cell_source(cell):
    # source 可能是 list[str] 或单字符串；统一拼成完整文本
    source = cell.get('source', [])
    return ''.join(source) if isinstance(source, list) else source

def set_cell_source(cell, new_source):
    # 若原格是 list 形式，写回时用 splitlines(keepends=True) 保持行列表风格
    if isinstance(cell.get('source', []), list):
        cell['source'] = new_source.splitlines(keepends=True)
    else:
        # 否则直接存整段字符串
        cell['source'] = new_source

def write_notebook(notebook, dest_path):
    """Write a notebook dict to disk."""
    # indent=1、ensure_ascii=False：接近常见 ipynb 落盘风格
    with open(dest_path, 'w', encoding='utf-8') as f:
        json.dump(notebook, f, indent=1, ensure_ascii=False)


In [ ]:
# ========== 扫描规则：哪些模式表示「直接提供商 API」，哪些必须跳过 ==========

# 指示直接提供者 API 使用的模式
CONVERSION_PATTERNS = [
    # 直接提供商客户端和 URL
    r'(?<!\w)OpenAI\(\)',
    r'api\.anthropic\.com',
    r'generativelanguage\.googleapis\.com',
    r'api\.x\.ai',
    r'api\.groq\.com',
    # 原生 SDK 导入
    r'from anthropic import',
    r'from google import genai',
    # 每个提供商的 API 密钥加载
    r"os\.getenv\(['\"]OPENAI_API_KEY['\"]\)",
    r"os\.getenv\(['\"]ANTHROPIC_API_KEY['\"]\)",
    r"os\.getenv\(['\"]GOOGLE_API_KEY['\"]\)",
    r"os\.getenv\(['\"]GROK_API_KEY['\"]\)",
    r"os\.getenv\(['\"]GROQ_API_KEY['\"]\)",
    r"os\.environ\[['\"]OPENAI_API_KEY['\"]\]",
    # 字典中的客户端变量引用 ({"gpt-5": openai, ...})
    r':\s*openai\b',
    r':\s*anthropic\b',
    r':\s*gemini\b',
    r':\s*grok\b',
    r':\s*groq\b',
    # 用作客户端对象的提供程序变量（openai.chat.completions...）
    r'\bopenai\.',
    r'\banthropic\.',
    r'\bgemini\.',
    r'\bgrok\.',
    r'\bgroq\.',
    # 作为函数参数传递的提供程序变量
    r'[\(,]\s*openai\b',
    r'[\(,]\s*anthropic\b',
    r'[\(,]\s*gemini\b',
    r'[\(,]\s*grok\b',
    r'[\(,]\s*groq\b',
    # 需要提供程序前缀的无前缀模型名称字符串
    r"""['"](?:gpt-|claude-|gemini-|grok-|deepseek-)[^'"]*['"]""",
]

# 硬跳过——这些单元格永远不应该被触及
SKIP_PATTERNS = [
    r'fine_tuning',
    r'images\.generate',
    r'audio\.speech',
]


def needs_conversion(cell_source):
    """Check if a code cell needs conversion.

    Returns True if the cell contains direct provider API usage (client setup,
    native SDK imports, per-provider key loading, or unprefixed model names).
    Returns False for cells using unsupported features (fine-tuning, DALL-E, TTS).
    """
    # 先看硬跳过：微调 / 图像 / 语音等 OpenRouter 不支持的能力
    for pattern in SKIP_PATTERNS:
        if re.search(pattern, cell_source):
            return False
    # 再看是否命中「直接提供商」特征
    for pattern in CONVERSION_PATTERNS:
        if re.search(pattern, cell_source):
            return True
    return False


def scan_notebook(notebook):
    # 遍历所有代码格，收集需要转换的 {index, source}
    results = []
    for i, cell in enumerate(notebook['cells']):
        if cell['cell_type'] != 'code':
            continue
        source = get_cell_source(cell)
        # 非空且 needs_conversion 为真才入队
        if source.strip() and needs_conversion(source):
            results.append({'index': i, 'source': source})
    return results


In [ ]:
# ========== SYSTEM_PROMPT：注入 MODEL_MAP，要求按 CELL 标记批量改写 ==========

# 批次分隔标记模板；{} 处填 batch 内序号
CELL_MARKER = '# ===CELL {}==='

# 构建一个可读的映射字符串以注入到提示中
map_str = '\n'.join(f'   "{k}" -> "{v}"' for k, v in MODEL_MAP.items())

# f-string system prompt：规则与 map 注入后发给模型（正文保持英文，影响转换行为）
SYSTEM_PROMPT = f"""You are a code conversion specialist. You will receive multiple Python code
cells separated by markers like `# ===CELL 0===`. Convert ALL cells to use OpenRouter.

RULES:
1. Replace ALL direct provider clients with a SINGLE OpenRouter client:
   `client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv('OPENROUTER_API_KEY'))`
2. Replace ALL separate API key variables (openai_api_key, anthropic_api_key, google_api_key,
   grok_api_key, groq_api_key) with one: openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
3. Use this MODEL_MAP for exact model name conversions:
{map_str}
   For any model NOT in this map:
   - "gpt-*" -> "openai/gpt-*"
   - "claude-*" -> "anthropic/claude-*"
   - "gemini-*" -> "google/gemini-*"
   - "grok-*" -> "x-ai/grok-*"
   - "deepseek-*" -> "deepseek/deepseek-*"
   - Already prefixed (contains "/") -> leave as-is
4. Replace ALL references to per-provider client variables (openai, anthropic, gemini,
   grok, groq) with the single `client` variable. This includes:
   - Client dictionaries: {{"gpt-5": openai, ...}} -> {{"openai/gpt-5": client, ...}}
   - Method calls: openai.chat.completions.create(...) -> client.chat.completions.create(...)
   - Function arguments: port(openai, MODEL, code) -> port(client, MODEL, code)
5. REMOVE Ollama/localhost models from model lists and client dicts entirely.
   Local model names (llama3.2, qwen2.5-coder, gpt-oss:20b, deepseek-r1:1.5b)
   are Ollama tags, not valid OpenRouter IDs. Drop them -- OpenRouter can't
   serve local models. Also remove any `ollama = OpenAI(base_url="http://localhost:...")` client setup.
6. PRESERVE all other logic, comments, function definitions, and structure
7. Keep the EXACT same `# ===CELL N===` markers in your output so I can split cells back
8. Fix incorrect provider prefixes in already-prefixed model names:
   - "xai/" -> "x-ai/"
   - "gemini/" -> "google/"

Respond ONLY with the converted code cells, keeping the markers. No explanations, no markdown fences."""


In [ ]:
# ========== 批量转换：一次 LLM 调用改多格，再按标记拆回 ==========

def convert_cells_batch(matches):
    """Convert all matched cells in one LLM call.
    
    Groups cells together so the LLM sees the full picture:
    key loading + client setup + model dicts as one unit.
    """
    # 把每格前插 CELL_MARKER，拼成一批文本
    parts = []
    for i, m in enumerate(matches):
        parts.append(CELL_MARKER.format(i))
        parts.append(m['source'])
    batch = '\n'.join(parts)

    # 非流式 Chat Completions：system 定规则，user 放待转换批次
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Convert these code cells to use OpenRouter:\n\n{batch}"}
        ]
    )
    # 取助手正文并去掉首尾空白
    result = response.choices[0].message.content.strip()
    # 剥离降价围栏
    result = re.sub(r'^```(?:python)?\s*\n?|```\s*$', '', result).strip()

    # 使用标记分裂回单个细胞
    converted = {}
    cell_splits = re.split(r'# ===CELL (\d+)===\n?', result)
    # split 后奇数位是序号、偶数位是代码；成对取出
    for j in range(1, len(cell_splits) - 1, 2):
        cell_idx = int(cell_splits[j])
        cell_code = cell_splits[j + 1].strip()
        # 只接受落在 matches 范围内的序号
        if cell_idx < len(matches):
            converted[cell_idx] = cell_code

    return converted


In [ ]:
# ========== 转换整本笔记本：扫描 → 批量改写 → 写回 deepcopy 副本 ==========

def convert_notebook(input_path):
    """Convert a notebook to use OpenRouter.

    Returns:
        (summary_str, modified_notebook, matches_with_diffs)
        The caller decides whether/when to write the result.
    """
    # 规范化为 Path
    input_path = Path(input_path)
    # 读入原笔记本
    notebook = read_notebook(input_path)
    # deepcopy：后续只改副本
    modified = copy.deepcopy(notebook)
    # 找出需要转换的代码格
    matches = scan_notebook(notebook)

    # 无需转换：提早返回（文案保持原样）
    if not matches:
        return "No cells need conversion.", None, []

    # 一批发给 LLM
    converted_map = convert_cells_batch(matches)

    diffs = []
    summary = [f"Converting {len(matches)} cells in {input_path.name}:"]

    # 按匹配顺序把转换结果写回对应 notebook cell index
    for batch_idx, m in enumerate(matches):
        nb_idx = m['index']
        original = m['source']

        if batch_idx in converted_map:
            converted = converted_map[batch_idx]
            set_cell_source(modified['cells'][nb_idx], converted)
            # 清空 outputs / execution_count：转换后旧输出可能失真
            modified['cells'][nb_idx]['outputs'] = []
            modified['cells'][nb_idx]['execution_count'] = None

            diffs.append({'cell': nb_idx, 'before': original, 'after': converted})
            summary.append(f"\n--- Cell {nb_idx} ---")
            summary.append(f"BEFORE:\n{original[:200]}")
            summary.append(f"AFTER:\n{converted[:200]}")
        else:
            # LLM 输出缺标记：记失败（文案保持原样）
            summary.append(f"\n--- Cell {nb_idx}: FAILED (marker not found in LLM output) ---")

    return '\n'.join(summary), modified, diffs


## Gradio 用户界面

从下拉列表选择一门课的笔记本，**Scan** 预览哪些单元格会变，**Convert** 查看完整前后差异，再 **Apply** 覆盖原文件（会自动创建 `.bak` 备份）。


In [ ]:
# ========== Gradio 回调：发现笔记本、扫描、转换、应用（带备份） ==========

# 当前笔记本所在目录；再上溯三级当作课程仓库根（含 week*）
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (NOTEBOOK_DIR / "../../../").resolve()


def discover_notebooks():
    """Scan PROJECT_ROOT/week*/ for course notebooks.

    Returns a list of (label, absolute_path) tuples sorted by week then filename.
    Excludes community-contributions/, solutions/, and .bak files.
    """
    results = []
    # 按 week* 目录名排序后遍历
    for week_dir in sorted(PROJECT_ROOT.glob("week*")):
        if not week_dir.is_dir():
            continue
        for nb in sorted(week_dir.glob("*.ipynb")):
            # 跳过 .bak（虽然 glob *.ipynb 通常碰不到）
            if nb.suffix == ".bak":
                continue
            # 下拉显示相对路径，值为绝对路径字符串
            rel = nb.relative_to(PROJECT_ROOT)
            results.append((str(rel), str(nb)))
    return results


def on_notebook_select(path):
    # 未选：提示并禁用 Scan/Convert 按钮
    if not path:
        return "Select a notebook above.", gr.update(interactive=False), gr.update(interactive=False)
    # 已选：显示路径，启用按钮
    return f"`{path}`", gr.update(interactive=True), gr.update(interactive=True)


def do_scan(path):
    # 扫描哪些格需要转换，用 Markdown 预览前 300 字符
    if not path:
        return "Select a notebook first."
    p = Path(path)
    notebook = read_notebook(p)
    matches = scan_notebook(notebook)
    if not matches:
        return f"No cells in `{p.name}` need conversion."
    lines = [f"Found **{len(matches)}** cells to convert in `{p.name}`:\n"]
    for m in matches:
        lines.append(f"**Cell {m['index']}**")
        lines.append(f"```python\n{m['source'][:300]}\n```\n")
    return '\n'.join(lines)


def do_convert(path):
    # 调 convert_notebook；成功则启用 Apply，并把副本塞进 State
    if not path:
        return "Select a notebook first.", None, None, gr.update(interactive=False)
    summary, modified, diffs = convert_notebook(path)
    if modified is None:
        return summary, None, None, gr.update(interactive=False)
    # 拼 before/after 文本供人工检查
    diff_lines = []
    for d in diffs:
        diff_lines.append(f"--- Cell {d['cell']} BEFORE ---\n{d['before']}")
        diff_lines.append(f"--- Cell {d['cell']} AFTER  ---\n{d['after']}")
    return '\n\n'.join(diff_lines), modified, path, gr.update(interactive=True)


def do_apply(converted_state, path_state):
    # 先备份再覆盖原文件
    if converted_state is None:
        return "Nothing to apply -- run Convert first."
    p = Path(path_state)
    if not p.exists():
        return f"File not found: {p}"
    backup_path = p.with_suffix(p.suffix + '.bak')
    shutil.copy2(p, backup_path)
    write_notebook(converted_state, p)
    return f"Done -- backup at {backup_path}"


In [ ]:
# ========== 组装 Gradio Blocks：下拉 + Scan/Convert/Apply 事件链 ==========

# 启动时扫描仓库里的 week*/*.ipynb 作为下拉选项
notebook_choices = discover_notebooks()

with gr.Blocks(title="OpenRouter Notebook Converter", theme=gr.themes.Soft()) as ui:
    # 标题与说明（UI 文案保持原样）
    gr.Markdown("# OpenRouter Notebook Converter\nConvert any course notebook to use OpenRouter with a single API key.")

    # State：暂存转换后的 notebook dict 与原路径，供 Apply 使用
    converted_nb = gr.State(value=None)
    original_path = gr.State(value=None)

    notebook_input = gr.Dropdown(
        choices=notebook_choices,
        label="Select a course notebook",
        interactive=True,
    )
    file_info = gr.Markdown("Select a notebook above.")

    with gr.Row():
        scan_btn = gr.Button("1. Scan", variant="secondary", interactive=False)
        convert_btn = gr.Button("2. Convert", variant="primary", interactive=False)

    scan_output = gr.Markdown()
    convert_output = gr.Textbox(label="Conversion diff (before / after)", lines=18, show_copy_button=True)

    apply_btn = gr.Button("3. Apply changes to original file", variant="stop", interactive=False)
    apply_output = gr.Textbox(label="Result", lines=1, interactive=False)

    # 事件：选文件 → 扫 → 转 → 应用
    notebook_input.change(on_notebook_select, inputs=notebook_input, outputs=[file_info, scan_btn, convert_btn])
    scan_btn.click(do_scan, inputs=notebook_input, outputs=scan_output)
    convert_btn.click(do_convert, inputs=notebook_input, outputs=[convert_output, converted_nb, original_path, apply_btn])
    apply_btn.click(do_apply, inputs=[converted_nb, original_path], outputs=apply_output)

# 启动 Gradio 服务
ui.launch()


## 使用注意

**会转换的内容：** `OpenAI()` 直接调用、Anthropic/Google/xAI/Groq 提供商 URL、原生 SDK、各家 API 密钥加载、引用提供商变量的客户端字典等。

**会跳过的内容：** 微调（fine-tuning）、DALL-E、TTS（OpenRouter 不支持）。

**Apply 前务必人工检查 Convert 输出**——LLM 生成的代码仍可能有漏改或误改。
